Paper: https://arxiv.org/abs/2311.11482

Source code: https://github.com/suzgunmirac/meta-prompting/tree/main

In [10]:
import os
import json
import re
import io
import time
from contextlib import redirect_stdout
from openai import OpenAI
from IPython.display import display, Markdown
from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"), override=True)

True

In [11]:
META_PROMPTING = """You are Meta-Expert, an extremely clever expert with the unique ability to collaborate with multiple experts (such as Expert Problem Solver, Expert Mathematician, Expert Essayist, etc.) to tackle any task and solve any complex problems. 
Some experts are adept at generating solutions, while others excel in verifying answers and providing valuable feedback.

Note that you also have special access to Expert Python, which has the unique ability to generate and execute Python code given natural-language instructions. 
Expert Python is highly capable of crafting code to perform complex calculations when given clear and precise directions. You might therefore want to use it especially for computational tasks.

As Meta-Expert, your role is to oversee the communication between the experts, effectively using their skills to answer a given question while applying your own critical thinking and verification abilities.

To communicate with a expert, type its name (e.g., "Expert Linguist" or "Expert Puzzle Solver"), followed by a colon ":", and then provide a detailed instruction enclosed within triple quotes. For example:

Expert Mathematician:
\"\"\"
You are a mathematics expert, specializing in the fields of geometry and algebra.
Compute the Euclidean distance between the points (-2, 5) and (3, 7).
\"\"\"

Ensure that your instructions are clear and unambiguous, and include all necessary information within the triple quotes. You can also assign personas to the experts (e.g., "You are a physicist specialized in...").

Interact with only one expert at a time, and break complex problems into smaller, solvable tasks if needed. Each interaction is treated as an isolated event, so include all relevant details in every call.

If you or an expert finds a mistake in another expert's solution, ask a new expert to review the details, compare both solutions, and give feedback. 
You can request an expert to redo their calculations or work, using input from other experts. 
Keep in mind that all experts, except yourself, have no memory! Therefore, always provide complete information in your instructions when contacting them. 
Since experts can sometimes make errors, seek multiple opinions or independently verify the solution if uncertain. Before providing a final answer, always consult an expert for confirmation. 
Ideally, obtain or verify the final solution with two independent experts. However, aim to present your final answer within 15 rounds or fewer.

Refrain from repeating the very same questions to experts. Examine their responses carefully and seek clarification if required, keeping in mind they don't recall past interactions.

Present the final answer as follows:
>> FINAL ANSWER:
\"\"\"
[final answer]
\"\"\"

For multiple-choice questions, select only one option. Each question has a unique answer, so analyze the provided information carefully to determine the most accurate and appropriate response. 
Please present only one solution if you come across multiple options.
"""

In [12]:
client = OpenAI()

# Utils (Replaces utils.execute_code)
def execute_python_code(code_text: str) -> str:
    """Simula 'execute_code_with_timeout' capturando la salida estándar."""
    f = io.StringIO()
    try:
        with redirect_stdout(f):
            exec(code_text, {})
        return f.getvalue()
    except Exception as e:
        return f"Error executing code: {str(e)}"
    
def generate_llm_response(messages: list) -> str:
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        temperature=0.1,
        # max_tokens=1024
    )
    return response.choices[0].message.content

In [13]:
EXPERT_PYTHON_MESSAGE = """You are an expert in Python and can generate Python code. 
To execute the code and display its output in the terminal using print statements, please make sure to include "Please run this code!" after the code block (i.e., after the closing code blocks)
"""
INTERMEDIATE_FEEDBACK = """Based on the information given, what are the most logical next steps or conclusions? Please make sure that the solution is accurate, directly answers the original question, and follows to all given constraints. 
Additionally, please review the final solution yourself or have another expert(s) verify it.
"""

TASK_INPUT = "2, 3, 4, 4"

TRIPLE_QUOTES = '"""'

meta_model_message_list = [{
            "role": "system", 
            "content": """
                You are an AI assistant that helps people find information. 
                Please answer the following question. Once you have determined the final answer, please present it using the format below:
                
                >> FINAL ANSWER:
                \"\"\"
                [final answer]
                \"\"\"
                """
            }]
error_message = "If you have determined the final answer, please present it using the format below:\n\n>> FINAL ANSWER:\n\"\"\"\n[final answer]\n\"\"\""
final_answer_indicator = ">> FINAL ANSWER:"

question_prefix = META_PROMPTING    

task_description = """
"Let's play a game called 24. 
You'll be given four integers, and your objective is to use each number only once, combined with any of the four arithmetic operations (addition, subtraction, multiplication, and division) and parentheses, to achieve a total of 24."

YOUR SPECIFIC INSTRUCTIONS:
1. Immediately consult 'Expert Python' to write and execute a script that finds a valid solution for these numbers.
2. The script should iterate through permutations of the numbers and operators.
3. Once the code is executed and returns a result, verify it and provide the final answer along with the code used.
"""

initial_user_content = f"""
{question_prefix}

Question: 
{task_description}

{TASK_INPUT}
"""

# Iniciar el historial de mensajes copiando la lista base del JSON
messages = meta_model_message_list.copy()
messages.append({
    "role": "user",
    "content": initial_user_content
})

print(initial_user_content)


You are Meta-Expert, an extremely clever expert with the unique ability to collaborate with multiple experts (such as Expert Problem Solver, Expert Mathematician, Expert Essayist, etc.) to tackle any task and solve any complex problems. 
Some experts are adept at generating solutions, while others excel in verifying answers and providing valuable feedback.

Note that you also have special access to Expert Python, which has the unique ability to generate and execute Python code given natural-language instructions. 
Expert Python is highly capable of crafting code to perform complex calculations when given clear and precise directions. You might therefore want to use it especially for computational tasks.

As Meta-Expert, your role is to oversee the communication between the experts, effectively using their skills to answer a given question while applying your own critical thinking and verification abilities.

To communicate with a expert, type its name (e.g., "Expert Linguist" or "Expe

In [14]:
def tag_round(content, round_number):
    print(f"[DEBUG] Tagging content for ROUND {round_number}")
    return f"\nROUND {round_number}:\n\n{content}"


def get_expert_name(text):
    name = text.strip().split("\n")[-1].replace(":", "").strip()
    print(f"[DEBUG] Extracted expert name: {name}")
    return name


def extract_code(text):
    print("[DEBUG] Extracting code block from expert output")

    text = text.replace("```python", "```")
    if "```" not in text:
        print("[WARN] No code block found, using raw text")
        return text.strip()

    parts = text.split("```")
    code = parts[-2].strip() if len(parts) >= 3 else parts[1].strip()

    print("[DEBUG] Code extracted successfully")
    return code


def execute_if_python(name, output):
    if name == "Expert Python" and "Please run this code!" in output:
        print("[INFO] Python expert requested code execution")

        code_part = output.split("Please run this code!")[0].strip()
        code = extract_code(code_part)

        print("[INFO] Executing Python code...")
        print(code)
        result = execute_python_code(code)
        print(f"[INFO] Execution completed: {result}")

        return (
            f"{output}\n\n"
            f"Here is the Python code used:\n\n{code}\n\n"
            f"Execution output:\n\n{result}"
        )

    print(f"[DEBUG] No execution needed for {name}")
    return (
        output
        .replace("* * *", "")
        .replace("FINAL ANSWER:", f"{name}'s final answer:\n")
    )

In [15]:
counter = 0
MAX_ROUNDS = 8

EXPERT_PATTERN = r"Expert ((?:\w+ ?){1,5}):\n"

while counter < MAX_ROUNDS:
    round_number = counter + 1
    print(f"\n[INFO] ===== STARTING ROUND {round_number} =====")

    # Tag round
    messages[-1]["content"] = tag_round(messages[-1]["content"], round_number)

    if counter == MAX_ROUNDS:
        print("[INFO] Marking as final round")
        messages[-1]["content"] += "\nThis is the last round; so, please present your final answer."

    # Call meta-model
    print("[INFO] Calling meta-model...")
    meta_output = generate_llm_response(messages)
    messages.append({"role": "assistant", "content": meta_output})

    print(f"[DEBUG] Meta-model input:")
    print(messages)
    print("\n\n")
    print(f"[DEBUG] Meta-model response:\n{meta_output}\n\n\n")

    # --- CASE 1: Experts detected ---
    if re.search(EXPERT_PATTERN, meta_output):
        print("[INFO] Expert invocation detected")

        parts = meta_output.split(TRIPLE_QUOTES)
        combined_output = ""

        for i in range(1, len(parts), 2):
            print(f"[DEBUG] Processing expert block {i//2 + 1}")

            preceding = parts[i - 1]
            content = parts[i]

            name = get_expert_name(preceding)
            print(f"[INFO] Invoking expert: {name}")

            instruction = f"You are {name}.\n\n{content.strip()}"

            if name == "Expert Python":
                print("[DEBUG] Applying Python expert system prompt")
                instruction = f"{EXPERT_PYTHON_MESSAGE}.\n\n{instruction}"

            expert_messages = [{"role": "user", "content": instruction}]

            print(f"[INFO] Calling expert model: {name}")
            expert_output = generate_llm_response(expert_messages)

            print(f"[DEBUG] Raw expert input:")
            print(expert_messages)
            print("\n\n")
            print(f"[DEBUG] Raw expert output:\n{expert_output}\n\n\n")

            expert_output = execute_if_python(name, expert_output)

            combined_output += (
                f"{name}'s output:\n"
                f"{TRIPLE_QUOTES}\n{expert_output}\n{TRIPLE_QUOTES}\n"
            )

        print("[INFO] Sending expert feedback back to meta-model")

        feedback = f"{combined_output.strip()}\n\n{INTERMEDIATE_FEEDBACK}"
        messages.append({"role": "user", "content": feedback})

    # --- CASE 2: Final answer ---
    elif final_answer_indicator in meta_output:
        print(f"[SUCCESS] Final answer found in round {round_number}")
        break

    # --- CASE 3: Bad format ---
    else:
        print("[WARNING] Incorrect format detected, requesting correction")
        messages.append({"role": "user", "content": error_message})

    counter += 1
    print(f"[DEBUG] Moving to next round (counter={counter})")

    time.sleep(1)


print("\n[INFO] ===== FINAL RESULT =====")

final_text = (
    messages[-1]["content"]
    if messages[-1]["role"] == "assistant"
    else messages[-2]["content"]
)

print("[INFO] Final output ready:\n")
print(final_text)


[INFO] ===== STARTING ROUND 1 =====
[DEBUG] Tagging content for ROUND 1
[INFO] Calling meta-model...
[DEBUG] Meta-model input:
[{'role': 'system', 'content': '\n                You are an AI assistant that helps people find information. \n                Please answer the following question. Once you have determined the final answer, please present it using the format below:\n\n                >> FINAL ANSWER:\n                """\n                [final answer]\n                """\n                '}, {'role': 'user', 'content': '\nROUND 1:\n\n\nYou are Meta-Expert, an extremely clever expert with the unique ability to collaborate with multiple experts (such as Expert Problem Solver, Expert Mathematician, Expert Essayist, etc.) to tackle any task and solve any complex problems. \nSome experts are adept at generating solutions, while others excel in verifying answers and providing valuable feedback.\n\nNote that you also have special access to Expert Python, which has the unique abil